In [ ]:
#Import packages 

import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import pandas as pd 
import json 
from math import cos, radians
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import seaborn as sns 
import matplotlib

import glob
import os
import csv
import ast
import calendar
from matplotlib.patches import Patch
from functools import reduce
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter

## Read in co-occurrence dfs 

### Outage Probability - TPL  

In [ ]:
all_outage = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/all_outage_rez_6km_TPL_rev.csv').drop(columns = ['Unnamed: 0'])

all_outage = all_outage[['season', 
                         'T_outage_prob', 
                         'P_outage_prob', 
                         'L_outage_prob', 
                         'TP_outage_prob', 
                         'TL_outage_prob',
                         'PL_outage_prob',
                         'TPL_outage_prob',
                         'outage_dur', 
                         'Pct']]

all_outage_all_seasons = all_outage[all_outage['season'] == 'All Seasons'].reset_index(drop=True)

In [ ]:
def restructure_outage_weather_probs_rev1(df):

    col_to_event = {
        'T_outage_prob': 'T',
        'P_outage_prob': 'P',
        'L_outage_prob': 'L',
        'TP_outage_prob':'TP',
        'TL_outage_prob':'TL',
        'PL_outage_prob':'PL',
        'TPL_outage_prob':'TPL',
    }

    # Melt the DataFrame to long format
    melted = df.melt(
        id_vars=['season', 'outage_dur', 'Pct'],
        value_vars=col_to_event.keys(),
        var_name='Probability_Column',
        value_name='Probability'
    )

    # Map the probability column names to event labels
    melted['Weather_Event'] = melted['Probability_Column'].map(col_to_event)

    # Reorder columns
    final = melted[['season', 'Weather_Event', 'Probability', 'outage_dur', 'Pct']]

    return final

In [ ]:
all_outage_all_seasons = restructure_outage_weather_probs_rev1(all_outage_all_seasons)
all_outage_all_seasons['Event'] = 'Outage'
all_outage_all_seasons['Probability'] = all_outage_all_seasons['Probability'] * 100

### filtered 
all_outage_all_seasons_filtered = all_outage_all_seasons.copy()

### *** Outage Probability - with wind (TPLW) *** 

In [ ]:
all_outage_wind = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/all_outage_rez_with_wind_rev.csv').drop(columns = ['Unnamed: 0'])

all_outage_wind = all_outage_wind[['season', 
                         'T_outage_prob', 
                         'W_outage_prob',
                         'L_outage_prob',
                         'TW_outage_prob', 
                         'PW_outage_prob', 
                         'PWL_outage_prob',
                         'PL_outage_prob', 
                         'TL_outage_prob', 
                         'WL_outage_prob', 
                         'outage_dur', 
                         'Pct']]

all_outage_all_seasons_wind = all_outage_wind[all_outage_wind['season'] == 'All Seasons'].reset_index(drop=True)

In [ ]:
def restructure_outage_weather_probs_rev1_WIND(df):

    col_to_event = {
        'T_outage_prob': 'T',
        'W_outage_prob':'W', 
        'L_outage_prob': 'L',
        'TW_outage_prob':'TW', 
        'PW_outage_prob':'PW', 
        'PWL_outage_prob':'PWL', 
        'PL_outage_prob':'PL',
        'TL_outage_prob':'TL',
        'WL_outage_prob':'WL', 
    }

    # Melt the DataFrame to long format
    melted = df.melt(
        id_vars=['season', 'outage_dur', 'Pct'],
        value_vars=col_to_event.keys(),
        var_name='Probability_Column',
        value_name='Probability'
    )

    # Map the probability column names to event labels
    melted['Weather_Event'] = melted['Probability_Column'].map(col_to_event)

    # Reorder columns
    final = melted[['season', 'Weather_Event', 'Probability', 'outage_dur', 'Pct']]

    return final

In [ ]:
all_outage_all_seasons_wind = restructure_outage_weather_probs_rev1_WIND(all_outage_all_seasons_wind)
all_outage_all_seasons_wind['Event'] = 'Outage'
all_outage_all_seasons_wind['Probability'] = all_outage_all_seasons_wind['Probability'] * 100


### filtered 
all_outage_all_seasons_wind_filtered = all_outage_all_seasons_wind.copy()

### Read in statistical significance df - TPL 

In [ ]:
outage_SIGNIFICANCE = (
    pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/aall_outage_SIGNIFICANCE_df_6km_TPL_rev.csv')
      .drop(columns=['Unnamed: 0'])
      .rename(columns={'Stars': 'sig'})
)

outage_SIGNIFICANCE['sig'] = outage_SIGNIFICANCE['sig'].fillna('')


## select cols 
outage_SIGNIFICANCE = outage_SIGNIFICANCE[[
    'Weather_Event', 
    'sig', 
    'Percentile', 
    'Duration (hr)'
]]

# weather event --> shorthand mapping 
mapping = {
    "Hot_Day_Only": "T",      
    "Rainy_Day_Only": "P", 
    "Extreme_Lightning_Day_Only": "L",  
    "PL_Only": "PL", 
    "TL_Only": "TL", 
    "TP_Only": "TP", 
    "TPL_Only": "TPL"
}

outage_SIGNIFICANCE["Weather_Event"] = outage_SIGNIFICANCE["Weather_Event"].map(mapping)
outage_SIGNIFICANCE = outage_SIGNIFICANCE.rename(columns = {'Duration (hr)':'outage_dur'})

### *** Read in significance df - TPLW 

In [ ]:
outage_SIGNIFICANCE_wind = (
    pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/aall_outage_SIGNIFICANCE_df_6km_with_wind_rev.csv')
      .drop(columns=['Unnamed: 0'])
      .rename(columns={'Stars': 'sig'})
)

outage_SIGNIFICANCE_wind['sig'] = outage_SIGNIFICANCE_wind['sig'].fillna('')


## select cols 
outage_SIGNIFICANCE_wind = outage_SIGNIFICANCE_wind[[
    'Weather_Event', 
    'sig', 
    'Percentile', 
    'Duration (hr)'
]]

# weather event --> shorthand mapping 
mapping = {
    "Hot_Day_Only": "T",  
    "Rainy_Day_Only": "P",   
    "Extreme_Lightning_Day_Only": "L",  
    "Windy_Day_Only": "W",      
    'TW_Only':'TW',
    'PW_Only':'PW',
    'WL_Only':'WL', 
    'PL_Only':'PL', 
    'TL_Only':'TL', 
    'PWL_Only':'PWL', 
    "TP_Only": "TP", 
    "TPL_Only": "TPL", 
    "TPW_Only": "TPW", 
    "TWL_Only": "TWL"
}

outage_SIGNIFICANCE_wind["Weather_Event"] = outage_SIGNIFICANCE_wind["Weather_Event"].map(mapping)
outage_SIGNIFICANCE_wind = outage_SIGNIFICANCE_wind.rename(columns = {'Duration (hr)':'outage_dur'})

## 1.1) Outage - 90th pct 

### Original df - TPL 

In [ ]:
df = all_outage_all_seasons_filtered


df_90 = df[df['Pct'] == '90th'].reset_index(drop=True)
weather_events_90 = ['T', 'P', 'PL', 'L', 'TL', 'TP']
df_90 = df_90[df_90['Weather_Event'].isin(weather_events_90)].reset_index(drop=True)


all_outage_all_seasons_filtered_90 = df_90.copy()

### With Wind - TPLW 

In [ ]:
df = all_outage_all_seasons_wind_filtered


df_wind_90 = df[df['Pct'] == '90th'].reset_index(drop=True)
weather_events_wind_90 = ['W', 'TW', 'PW', 'PWL']
df_wind_90 = df_wind_90[df_wind_90['Weather_Event'].isin(weather_events_wind_90)].reset_index(drop=True)


all_outage_all_seasons_wind_filtered_90 = df_wind_90.copy()

### *** Combined co-occurrence dfs (90th pct) 

In [ ]:
dfs = [all_outage_all_seasons_filtered_90, all_outage_all_seasons_wind_filtered_90]

all_outage_all_seasons_filtered_90_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Outage significance (90th) - *TPL* 

In [ ]:
outage_SIGNIFICANCE_90 = outage_SIGNIFICANCE[outage_SIGNIFICANCE['Percentile'] == '90th'].reset_index(drop=True)

outage_SIGNIFICANCE_90 = outage_SIGNIFICANCE_90[outage_SIGNIFICANCE_90['Weather_Event'].isin(weather_events_90)].reset_index(drop=True)

### Outage significance (90th) - *TPLW* 

In [ ]:
outage_SIGNIFICANCE_wind_90 = outage_SIGNIFICANCE_wind[outage_SIGNIFICANCE_wind['Percentile'] == '90th'].reset_index(drop=True)

outage_SIGNIFICANCE_wind_90 = outage_SIGNIFICANCE_wind_90[outage_SIGNIFICANCE_wind_90['Weather_Event'].isin(weather_events_wind_90)].reset_index(drop=True)

### *** COMBINED significance (90th pct) 

In [ ]:
dfs = [outage_SIGNIFICANCE_90, outage_SIGNIFICANCE_wind_90]
outage_significance_90_combined = pd.concat(dfs, axis = 0, ignore_index=True) 

### Tack on the significance stars & re-order (90th pct) 

In [ ]:
# Do a merge on 'Weather_Event' & outage_dur

all_outage_all_seasons_filtered_90_combined = pd.merge(
    all_outage_all_seasons_filtered_90_combined, 
    outage_significance_90_combined[['Weather_Event', 'sig', 'outage_dur']], 
    on=['Weather_Event', 'outage_dur'], 
    how='inner')



order = ['T', 'W', 'TW', 'P', 'PL', 'L', 'PW', 'TL', 'TP', 'PWL']


df = all_outage_all_seasons_filtered_90_combined

df['Weather_Event'] = pd.Categorical(
    df['Weather_Event'],
    categories=order,
    ordered=True
)

df = (
    df
    .sort_values(by=['outage_dur', 'Weather_Event'])
    .reset_index(drop=True)
)

all_outage_all_seasons_filtered_90_combined = df.copy()

## 1.2) Outage - 95th pct 

### Original df - TPL 

In [ ]:
df = all_outage_all_seasons_filtered


df_95 = df[df['Pct'] == '95th'].reset_index(drop=True)
weather_events_95 = ['T', 'P', 'PL', 'TL']
df_95 = df_95[df_95['Weather_Event'].isin(weather_events_95)].reset_index(drop=True)


all_outage_all_seasons_filtered_95 = df_95.copy()

### With wind - TPLW 

In [ ]:
df = all_outage_all_seasons_wind_filtered


df_wind_95 = df[df['Pct'] == '95th'].reset_index(drop=True)
weather_events_wind_95 = ['W', 'L', 'TW', 'PWL', 'PW', 'WL']
df_wind_95 = df_wind_95[df_wind_95['Weather_Event'].isin(weather_events_wind_95)].reset_index(drop=True)


all_outage_all_seasons_wind_filtered_95 = df_wind_95.copy()

### *** Combined co-occurrence dfs (95th pct)

In [ ]:
dfs = [all_outage_all_seasons_filtered_95, all_outage_all_seasons_wind_filtered_95]

all_outage_all_seasons_filtered_95_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Outage significance (95th) - TPL 

In [ ]:
outage_SIGNIFICANCE_95 = outage_SIGNIFICANCE[outage_SIGNIFICANCE['Percentile'] == '95th'].reset_index(drop=True)

outage_SIGNIFICANCE_95 = outage_SIGNIFICANCE_95[outage_SIGNIFICANCE_95['Weather_Event'].isin(weather_events_95)].reset_index(drop=True)

### Outage significance (95th) - TPLW 

In [ ]:
outage_SIGNIFICANCE_wind_95 = outage_SIGNIFICANCE_wind[outage_SIGNIFICANCE_wind['Percentile'] == '95th'].reset_index(drop=True)

outage_SIGNIFICANCE_wind_95 = outage_SIGNIFICANCE_wind_95[outage_SIGNIFICANCE_wind_95['Weather_Event'].isin(weather_events_wind_95)].reset_index(drop=True)

### *** COMBINED significance (95th pct) 

In [ ]:
dfs = [outage_SIGNIFICANCE_95, outage_SIGNIFICANCE_wind_95]

outage_significance_95_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Tack on the significance stars  

In [ ]:
all_outage_all_seasons_filtered_95_combined = pd.merge(
    all_outage_all_seasons_filtered_95_combined, 
    outage_significance_95_combined[['Weather_Event', 'sig', 'outage_dur']], 
    on=['Weather_Event', 'outage_dur'], 
    how='inner')


order = ['T', 'W', 'P', 'L', 'PL', 'TW', 'PWL', 'PW', 'TL', 'WL']


df = all_outage_all_seasons_filtered_95_combined

df['Weather_Event'] = pd.Categorical(
    df['Weather_Event'],
    categories=order,
    ordered=True
)

df = (
    df
    .sort_values(by=['outage_dur', 'Weather_Event'])
    .reset_index(drop=True)
)


all_outage_all_seasons_filtered_95_combined = df.copy()

## 1.3) Outage - 99th pct 

### Original df - TPL 

In [ ]:
df = all_outage_all_seasons_filtered


df_99 = df[df['Pct'] == '99th'].reset_index(drop=True)
weather_events_99 = ['P']
df_99 = df_99[df_99['Weather_Event'].isin(weather_events_99)].reset_index(drop=True)

all_outage_all_seasons_filtered_99 = df_99.copy()

### With wind - TPLW 

In [ ]:
df = all_outage_all_seasons_wind_filtered


df_wind_99 = df[df['Pct'] == '99th'].reset_index(drop=True)
weather_events_wind_99 = ['L', 'T', 'PL', 'W', 'WL', 'PW', 'TL', 'PWL', 'TW']
df_wind_99 = df_wind_99[df_wind_99['Weather_Event'].isin(weather_events_wind_99)].reset_index(drop=True)

all_outage_all_seasons_wind_filtered_99 = df_wind_99.copy()

### *** Combined dfs (99th pct) 

In [ ]:
dfs = [all_outage_all_seasons_filtered_99, all_outage_all_seasons_wind_filtered_99]

all_outage_all_seasons_filtered_99_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Outage significance (99th) - TPL 

In [ ]:
outage_SIGNIFICANCE_99 = outage_SIGNIFICANCE[outage_SIGNIFICANCE['Percentile'] == '99th'].reset_index(drop=True)

outage_SIGNIFICANCE_99 = outage_SIGNIFICANCE_99[outage_SIGNIFICANCE_99['Weather_Event'].isin(weather_events_99)].reset_index(drop=True)

### Outage significance (99th) - TPLW 

In [ ]:
outage_SIGNIFICANCE_wind_99 = outage_SIGNIFICANCE_wind[outage_SIGNIFICANCE_wind['Percentile'] == '99th'].reset_index(drop=True)

outage_SIGNIFICANCE_wind_99 = outage_SIGNIFICANCE_wind_99[outage_SIGNIFICANCE_wind_99['Weather_Event'].isin(weather_events_wind_99)].reset_index(drop=True)

### *** COMBINED significance (99th pct) 

In [ ]:
dfs = [outage_SIGNIFICANCE_99, outage_SIGNIFICANCE_wind_99]

outage_significance_99_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Tack on the significance stars  

In [ ]:
all_outage_all_seasons_filtered_99_combined = pd.merge(
    all_outage_all_seasons_filtered_99_combined, 
    outage_significance_99_combined[['Weather_Event', 'sig', 'outage_dur']], 
    on=['Weather_Event', 'outage_dur'], 
    how='inner')


order = ['P', 'L', 'T', 'PL', 'W', 'WL', 'PW', 'TL', 'PWL', 'TW']


df = all_outage_all_seasons_filtered_99_combined

df['Weather_Event'] = pd.Categorical(
    df['Weather_Event'],
    categories=order,
    ordered=True
)

df = (
    df
    .sort_values(by=['outage_dur', 'Weather_Event'])
    .reset_index(drop=True)
)


all_outage_all_seasons_filtered_99_combined = df.copy()

## Undervoltages 

### Undervoltage Probability - TPL 

In [ ]:
all_undervolt = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/all_undervolt_df_6km_TPL_rev.csv').drop(columns = ['Unnamed: 0'])

all_undervolt = all_undervolt[['season', 
                         'T_undervolt_prob', 
                         'P_undervolt_prob', 
                         'L_undervolt_prob', 
                         'TP_undervolt_prob',
                         'TL_undervolt_prob',
                         'PL_undervolt_prob',
                         'TPL_undervolt_prob', 
                         'undervolt_dur', 
                         'Pct']]

all_undervolt_all_seasons = all_undervolt[all_undervolt['season'] == 'All Seasons'].reset_index(drop=True)

In [ ]:
def restructure_undervolt_weather_probs_rev1(df):
    # Mapping from probability column names to Weather_Event labels
    col_to_event = {
        'T_undervolt_prob': 'T',
        'P_undervolt_prob': 'P',
        'L_undervolt_prob': 'L',
        'TP_undervolt_prob':'TP', 
        'TL_undervolt_prob':'TL', 
        'PL_undervolt_prob':'PL', 
        'TPL_undervolt_prob': 'TPL'
    }

    # Melt the DataFrame to long format
    melted = df.melt(
        id_vars=['season', 'undervolt_dur', 'Pct'],
        value_vars=col_to_event.keys(),
        var_name='Probability_Column',
        value_name='Probability'
    )

    # Map the probability column names to event labels
    melted['Weather_Event'] = melted['Probability_Column'].map(col_to_event)

    # Reorder columns
    final = melted[['season', 'Weather_Event', 'Probability', 'undervolt_dur', 'Pct']]

    return final

In [ ]:
all_undervolt_all_seasons = restructure_undervolt_weather_probs_rev1(all_undervolt_all_seasons)
all_undervolt_all_seasons['Event'] = 'Undervolt'
all_undervolt_all_seasons['Probability'] = all_undervolt_all_seasons['Probability'] * 100
all_undervolt_all_seasons['undervolt_dur'] = all_undervolt_all_seasons['undervolt_dur'] / 60

### filtered 
all_undervolt_all_seasons_filtered = all_undervolt_all_seasons.copy()

### Undervoltage Probability (with wind) - TPLW 

In [ ]:
all_undervolt_wind = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/all_undervolt_df_with_wind_rev.csv').drop(columns = ['Unnamed: 0'])


all_undervolt_wind = all_undervolt_wind[['season', 
                         'T_undervolt_prob', 
                         'P_undervolt_prob', 
                         'W_undervolt_prob', 
                         'L_undervolt_prob',
                         'TW_undervolt_prob', 
                         'TL_undervolt_prob', 
                         'PW_undervolt_prob',
                         'PL_undervolt_prob',
                         'WL_undervolt_prob', 
                         'PWL_undervolt_prob', 
                         # 'TP_undervolt_prob',  
                         'undervolt_dur', 
                         'Pct']]

all_undervolt_all_seasons_wind = all_undervolt_wind[all_undervolt_wind['season'] == 'All Seasons'].reset_index(drop=True)

In [ ]:
def restructure_undervolt_weather_probs_rev1_WIND(df):
    # Mapping from probability column names to Weather_Event labels
    col_to_event = {
        'T_undervolt_prob': 'T',
        'P_undervolt_prob':'P', 
        'W_undervolt_prob':'W', 
        'L_undervolt_prob': 'L',
        'TW_undervolt_prob':'TW', 
        'TL_undervolt_prob':'TL',
        'PW_undervolt_prob':'PW', 
        'PL_undervolt_prob':'PL',
        'WL_undervolt_prob':'WL', 
        'PWL_undervolt_prob':'PWL', 
        # 'TP_undervolt_prob':'TP'
    }

    # Melt the DataFrame to long format
    melted = df.melt(
        id_vars=['season', 'undervolt_dur', 'Pct'],
        value_vars=col_to_event.keys(),
        var_name='Probability_Column',
        value_name='Probability'
    )

    # Map the probability column names to event labels
    melted['Weather_Event'] = melted['Probability_Column'].map(col_to_event)

    # Reorder columns
    final = melted[['season', 'Weather_Event', 'Probability', 'undervolt_dur', 'Pct']]

    return final

In [ ]:
all_undervolt_all_seasons_wind = restructure_undervolt_weather_probs_rev1_WIND(all_undervolt_all_seasons_wind)
all_undervolt_all_seasons_wind['Event'] = 'Undervolt'
all_undervolt_all_seasons_wind['Probability'] = all_undervolt_all_seasons_wind['Probability'] * 100
all_undervolt_all_seasons_wind['undervolt_dur'] = all_undervolt_all_seasons_wind['undervolt_dur'] / 60

### filtered 
all_undervolt_all_seasons_wind_filtered = all_undervolt_all_seasons_wind.copy()

### Read in undervolt SIGNIFICANCE df - TPL 

In [ ]:
undervolt_SIGNIFICANCE = (
    pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/all_undervolt_SIGNIFICANCE_df_6km_TPL_rev.csv')
      .drop(columns=['Unnamed: 0'])
      .rename(columns={'Stars': 'sig'})
)

undervolt_SIGNIFICANCE['sig'] = undervolt_SIGNIFICANCE['sig'].fillna('')


## select cols 
undervolt_SIGNIFICANCE = undervolt_SIGNIFICANCE[[
    'Weather_Event', 
    'sig', 
    'Percentile', 
    'Duration (hr)'
]]

# weather event --> shorthand mapping 
mapping = {
    "Hot_Day_Only": "T",      
    "Rainy_Day_Only": "P",     
    "Extreme_Lightning_Day_Only": "L",  
    "PL_Only": "PL", 
    "TL_Only": "TL", 
    "TP_Only": "TP", 
    "TPL_Only": "TPL"
}

undervolt_SIGNIFICANCE["Weather_Event"] = undervolt_SIGNIFICANCE["Weather_Event"].map(mapping)
undervolt_SIGNIFICANCE = undervolt_SIGNIFICANCE.rename(columns = {'Duration (hr)':'undervolt_dur'})

### *** Read in undervolt significance df - TPLW 

In [ ]:
undervolt_SIGNIFICANCE_wind = (
    pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/all_undervolt_SIGNIFICANCE_df_wind_6km_with_wind_rev.csv')
      .drop(columns=['Unnamed: 0'])
      .rename(columns={'Stars': 'sig'})
)

undervolt_SIGNIFICANCE_wind['sig'] = undervolt_SIGNIFICANCE_wind['sig'].fillna('')


## select cols 
undervolt_SIGNIFICANCE_wind = undervolt_SIGNIFICANCE_wind[[
    'Weather_Event', 
    'sig', 
    'Percentile', 
    'Duration (hr)'
]]


# weather event --> shorthand mapping 
mapping = {
    "Hot_Day_Only": "T",  
    "Windy_Day_Only": "W",      
    "Rainy_Day_Only": "P",     
    "Extreme_Lightning_Day_Only": "L",  
    'TL_Only':'TL', 
    "TP_Only": "TP", 
    'TW_Only':'TW',
    'PL_Only':'PL', 
    'PW_Only':'PW',    
    'WL_Only':'WL', 
    "TPL_Only": "TPL",  
    "TPW_Only": "TPW", 
    "TWL_Only": "TWL",  
    'PWL_Only':'PWL', 
    'TPWL_Only':'TPWL', 
}

undervolt_SIGNIFICANCE_wind["Weather_Event"] = undervolt_SIGNIFICANCE_wind["Weather_Event"].map(mapping)
undervolt_SIGNIFICANCE_wind = undervolt_SIGNIFICANCE_wind.rename(columns = {'Duration (hr)':'undervolt_dur'})

## 2.1) Undervolt - 90th pct 

### Original df - TPL 

In [ ]:
df = all_undervolt_all_seasons_filtered

df_90 = df[df['Pct'] == '90th'].reset_index(drop=True)
weather_events_90 = ['T', 'P', 'PL', 'L', 'TL', 'TP']
df_90 = df_90[df_90['Weather_Event'].isin(weather_events_90)].reset_index(drop=True)

all_undervolt_all_seasons_filtered_90 = df_90.copy()

### With Wind - TPLW 

In [ ]:
df = all_undervolt_all_seasons_wind_filtered

df_wind_90 = df[df['Pct'] == '90th'].reset_index(drop=True)
weather_events_wind_90 = ['W', 'TW', 'PW', 'PWL']
df_wind_90 = df_wind_90[df_wind_90['Weather_Event'].isin(weather_events_wind_90)].reset_index(drop=True)

all_undervolt_all_seasons_wind_filtered_90 = df_wind_90.copy()

### *** Combined dfs (90th pct) 

In [ ]:
dfs = [all_undervolt_all_seasons_filtered_90, all_undervolt_all_seasons_wind_filtered_90]

all_undervolt_all_seasons_filtered_90_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Undervolt significance (90th) - TPL 

In [ ]:
undervolt_SIGNIFICANCE_90 = undervolt_SIGNIFICANCE[undervolt_SIGNIFICANCE['Percentile'] == '90th'].reset_index(drop=True)

undervolt_SIGNIFICANCE_90 = undervolt_SIGNIFICANCE_90[undervolt_SIGNIFICANCE_90['Weather_Event'].isin(weather_events_90)].reset_index(drop=True)

### Undervolt significance (90th) - TPLW 

In [ ]:
undervolt_SIGNIFICANCE_wind_90 = undervolt_SIGNIFICANCE_wind[undervolt_SIGNIFICANCE_wind['Percentile'] == '90th'].reset_index(drop=True)

undervolt_SIGNIFICANCE_wind_90 = undervolt_SIGNIFICANCE_wind_90[undervolt_SIGNIFICANCE_wind_90['Weather_Event'].isin(weather_events_wind_90)].reset_index(drop=True)

### *** Combined significance (90th pct) 

In [ ]:
dfs = [undervolt_SIGNIFICANCE_90, undervolt_SIGNIFICANCE_wind_90]

undervolt_significance_90_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Tack on the significance stars  

In [ ]:
all_undervolt_all_seasons_filtered_90_combined = pd.merge(
    all_undervolt_all_seasons_filtered_90_combined, 
    undervolt_significance_90_combined[['Weather_Event', 'sig', 'undervolt_dur']], 
    on=['Weather_Event', 'undervolt_dur'], 
    how='inner')


order = ['T', 'W', 'TW', 'P', 'PL', 'L', 'PW', 'TL', 'TP', 'PWL']


df = all_undervolt_all_seasons_filtered_90_combined

df['Weather_Event'] = pd.Categorical(
    df['Weather_Event'],
    categories=order,
    ordered=True
)

df = (
    df
    .sort_values(by=['undervolt_dur', 'Weather_Event'])
    .reset_index(drop=True)
)

all_undervolt_all_seasons_filtered_90_combined = df.copy()

## 2.2) Undervolt - 95th pct 

### Original df - TPL 

In [ ]:
df = all_undervolt_all_seasons_filtered

df_95 = df[df['Pct'] == '95th'].reset_index(drop=True)
weather_events_95 = ['T', 'P', 'PL', 'TL']
df_95 = df_95[df_95['Weather_Event'].isin(weather_events_95)].reset_index(drop=True)

all_undervolt_all_seasons_filtered_95 = df_95.copy()

### With Wind - TPLW 

In [ ]:
df = all_undervolt_all_seasons_wind_filtered

df_wind_95 = df[df['Pct'] == '95th'].reset_index(drop=True)
weather_events_wind_95 = ['W', 'L', 'TW', 'PWL', 'PW', 'WL']
df_wind_95 = df_wind_95[df_wind_95['Weather_Event'].isin(weather_events_wind_95)].reset_index(drop=True)

all_undervolt_all_seasons_wind_filtered_95 = df_wind_95.copy()

### *** Combined dfs (95th pct) 

In [ ]:
dfs = [all_undervolt_all_seasons_filtered_95, all_undervolt_all_seasons_wind_filtered_95]

all_undervolt_all_seasons_filtered_95_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Undervolt significance (95th) - TPL 

In [ ]:
undervolt_SIGNIFICANCE_95 = undervolt_SIGNIFICANCE[undervolt_SIGNIFICANCE['Percentile'] == '95th'].reset_index(drop=True)

undervolt_SIGNIFICANCE_95 = undervolt_SIGNIFICANCE_95[undervolt_SIGNIFICANCE_95['Weather_Event'].isin(weather_events_95)].reset_index(drop=True)

### Undervolt significance (95th) - TPLW 

In [ ]:
undervolt_SIGNIFICANCE_wind_95 = undervolt_SIGNIFICANCE_wind[undervolt_SIGNIFICANCE_wind['Percentile'] == '95th'].reset_index(drop=True)

undervolt_SIGNIFICANCE_wind_95 = undervolt_SIGNIFICANCE_wind_95[undervolt_SIGNIFICANCE_wind_95['Weather_Event'].isin(weather_events_wind_95)].reset_index(drop=True)

### *** Combined significance (95th pct) 

In [ ]:
dfs = [undervolt_SIGNIFICANCE_95, undervolt_SIGNIFICANCE_wind_95]

undervolt_significance_95_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Tack on the significance stars  

In [ ]:
all_undervolt_all_seasons_filtered_95_combined = pd.merge(
    all_undervolt_all_seasons_filtered_95_combined, 
    undervolt_significance_95_combined[['Weather_Event', 'sig', 'undervolt_dur']], 
    on=['Weather_Event', 'undervolt_dur'], 
    how='inner')



order = ['T', 'W', 'P', 'L', 'PL', 'TW', 'PWL', 'PW', 'TL', 'WL']


df = all_undervolt_all_seasons_filtered_95_combined

df['Weather_Event'] = pd.Categorical(
    df['Weather_Event'],
    categories=order,
    ordered=True
)

df = (
    df
    .sort_values(by=['undervolt_dur', 'Weather_Event'])
    .reset_index(drop=True)
)

all_undervolt_all_seasons_filtered_95_combined = df.copy()

## 2.3) Undervolt - 99th pct 

### Original df - TPL 

In [ ]:
df = all_undervolt_all_seasons_filtered

df_99 = df[df['Pct'] == '99th'].reset_index(drop=True)
weather_events_99 = ['P']
df_99 = df_99[df_99['Weather_Event'].isin(weather_events_95)].reset_index(drop=True)

all_undervolt_all_seasons_filtered_99 = df_99.copy()

### With Wind - TPLW 

In [ ]:
df = all_undervolt_all_seasons_wind_filtered

df_wind_99 = df[df['Pct'] == '99th'].reset_index(drop=True)
weather_events_wind_99 = ['L', 'T', 'PL', 'W', 'WL', 'PW', 'TL', 'PWL', 'TW']
df_wind_99 = df_wind_99[df_wind_99['Weather_Event'].isin(weather_events_wind_99)].reset_index(drop=True)

all_undervolt_all_seasons_wind_filtered_99 = df_wind_99.copy()

### *** Combined co-occurrence dfs (99th pct) 

In [ ]:
dfs = [all_undervolt_all_seasons_filtered_99, all_undervolt_all_seasons_wind_filtered_99]

all_undervolt_all_seasons_filtered_99_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Undervolt significance (99th) - TPL 

In [ ]:
undervolt_SIGNIFICANCE_99 = undervolt_SIGNIFICANCE[undervolt_SIGNIFICANCE['Percentile'] == '99th'].reset_index(drop=True)

undervolt_SIGNIFICANCE_99 = undervolt_SIGNIFICANCE_99[undervolt_SIGNIFICANCE_99['Weather_Event'].isin(weather_events_99)].reset_index(drop=True)

### Undervolt significance (99th) - TPLW 

In [ ]:
undervolt_SIGNIFICANCE_wind_99 = undervolt_SIGNIFICANCE_wind[undervolt_SIGNIFICANCE_wind['Percentile'] == '99th'].reset_index(drop=True)

undervolt_SIGNIFICANCE_wind_99 = undervolt_SIGNIFICANCE_wind_99[undervolt_SIGNIFICANCE_wind_99['Weather_Event'].isin(weather_events_wind_99)].reset_index(drop=True)

### *** Combined significance (99th pct) 

In [ ]:
dfs = [undervolt_SIGNIFICANCE_99, undervolt_SIGNIFICANCE_wind_99]

undervolt_significance_99_combined = pd.concat(dfs, axis = 0, ignore_index=True)

### Tack on the significance stars  

In [ ]:
all_undervolt_all_seasons_filtered_99_combined = pd.merge(
    all_undervolt_all_seasons_filtered_99_combined, 
    undervolt_significance_99_combined[['Weather_Event', 'sig', 'undervolt_dur']], 
    on=['Weather_Event', 'undervolt_dur'], 
    how='inner')


order = ['P', 'L', 'T', 'PL', 'W', 'WL', 'PW', 'TL', 'PWL', 'TW']


df = all_undervolt_all_seasons_filtered_99_combined

df['Weather_Event'] = pd.Categorical(
    df['Weather_Event'],
    categories=order,
    ordered=True
)

df = (
    df
    .sort_values(by=['undervolt_dur', 'Weather_Event'])
    .reset_index(drop=True)
)

all_undervolt_all_seasons_filtered_99_combined = df.copy()

# ***** Plots ***** 

In [ ]:
def plot_grid_disturbance_probability_per_percentile_NY_times(
    dfs,
    titles,
    plot_title=None,
    weather_event_order=None,
    figsize=(18, 6), 
    xlim=(0, 85), 
    alpha=0.8, 
    x_label='Probability (%)',
    show_shared_xlabel=True,
    show_sig=True, 
    show_sig_legend=True,  
    save_path=None,
    duration_legend_pos='lower right',
    duration_legend_anchor=None,
    sig_legend_pos='center right',
    sig_legend_anchor=(1.05, 0.5),
    vline_pos=(10, 20, 30, 40, 50, 60, 70, 80, 90),
    vline_kwargs={'color':'white', 'linewidth':1.4, 'linestyle':'-'}, 
    hline=True,
    hline_kwargs={'color':'lightgrey', 'linewidth':1.4, 'linestyle':'-'}
):

    n_panels = len(dfs)
    fig, axs = plt.subplots(1, n_panels, figsize=figsize, sharex=True, constrained_layout=True)
    if n_panels == 1:
        axs = [axs]
        
    hatch_patterns = ['///', '']

    for i, (df, ax, title) in enumerate(zip(dfs, axs, titles)):
        # Identify disturbance type
        if "outage_dur" in df.columns:
            duration_col = "outage_dur"
            duration_order = [8, 1]
            duration_unit = "Outage"
            bar_color = "#918C8C"
        elif "undervolt_dur" in df.columns:
            duration_col = "undervolt_dur"
            duration_order = [4, 1]
            duration_unit = "Undervoltage"
            bar_color = "#D95D5D"
        elif "overvolt_dur" in df.columns:
            duration_col = "overvolt_dur"
            duration_order = [60, 20]
            duration_unit = "Overvoltage"
            bar_color = "#5D8AD9"
        else:
            raise ValueError("Unknown dataframe type")

        # Duration labels
        if duration_unit in ["Outage", "Undervoltage"]:
            duration_labels = {dur: f"{dur}+ hr" for dur in duration_order}
        else:  # Overvoltage
            duration_labels = {dur: f"{dur//60}+ hr" if dur >= 60 else f"{dur}+ min" for dur in duration_order}

        duration_hatches = {dur: hatch_patterns[j % len(hatch_patterns)] for j, dur in enumerate(duration_order)}

        # Weather event order
        if weather_event_order is None:
            weather_event_order = sorted(df["Weather_Event"].unique())

        # Pivot table
        pivot = df.pivot_table(
            index="Weather_Event",
            columns=duration_col,
            values="Probability",
            fill_value=0,
            observed=False
        ).reindex(index=weather_event_order, columns=duration_order)

        num_events = len(pivot)
        num_durations = len(duration_order)
        bar_height = 0.8 / num_durations
        y_base = np.arange(num_events) * 1.2

        # Plot bars
        for j, duration in enumerate(duration_order):
            y_offset = y_base + (j - num_durations / 2) * bar_height + bar_height / 2
            ax.barh(
                y=y_offset,
                width=pivot[duration],
                height=bar_height,
                color=bar_color,
                edgecolor="black",
                hatch=duration_hatches.get(duration, ""),
                label=duration_labels[duration],
                alpha=alpha
            )

            # Add significance stars
            if show_sig:
                for idx, we in enumerate(pivot.index):
                    sig_val = df.loc[(df["Weather_Event"] == we) & (df[duration_col] == duration), "sig"].values
                    if len(sig_val) > 0 and sig_val[0] != '':
                        ax.text(
                            pivot.loc[we, duration] + 1,
                            y_offset[idx] - 0.2*bar_height,
                            sig_val[0],
                            va='center',
                            fontsize=10,
                            fontweight='bold'
                        )

        # Vertical lines
        if vline_pos is not None:
            if isinstance(vline_pos, (int, float)):
                vline_pos = [vline_pos]
            for xpos in vline_pos:
                ax.axvline(x=xpos, **vline_kwargs)

        # Horizontal lines below each weather event group
        if hline:
            for y in y_base:
                y_bottom = y - (num_durations / 2) * bar_height - 0.14
                ax.axhline(y=y_bottom, **hline_kwargs)

        # Y-axis
        ax.set_yticks(y_base)
        if i == 0:
            ax.set_yticklabels(pivot.index, fontsize=12, fontweight="bold")
            ax.tick_params(axis="y", labelcolor="black", labelsize=15, width=1)
        else:
            ax.set_yticklabels([])

        ax.set_title(title, fontsize=14, fontweight="bold", pad=15)
        ax.tick_params(axis="x", labelsize=15)
        if xlim is not None:
            ax.set_xlim(xlim)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}%'))
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Duration legend
        handles, labels = ax.get_legend_handles_labels()
        dur_legend = ax.legend(
            handles[::-1],
            labels[::-1],
            fontsize=12,
            title_fontsize=14,
            title=f"{duration_unit} Duration",
            loc=duration_legend_pos,
            bbox_to_anchor=duration_legend_anchor
        )
        ax.add_artist(dur_legend)

    # Significance legend
    if show_sig and show_sig_legend:
        sig_patches = [
            mpatches.Patch(facecolor='none', edgecolor='none', label='*   p < 0.1'),
            mpatches.Patch(facecolor='none', edgecolor='none', label='**  p < 0.05'),
            mpatches.Patch(facecolor='none', edgecolor='none', label='*** p < 0.01')
        ]
        fig.legend(
            handles=sig_patches,
            labels=[p.get_label() for p in sig_patches],
            title="Significance levels:",
            title_fontsize=15,
            frameon=False,
            prop={'family': 'monospace', 'size': 14},
            loc=sig_legend_pos,
            bbox_to_anchor=sig_legend_anchor
        )

    # Main title
    if plot_title:
        fig.suptitle(plot_title, fontsize=16, fontweight="bold", y=1.05)

    # Shared X and Y labels
    if show_shared_xlabel:
        fig.text(0.5, -0.08, x_label, ha="center", fontsize=18, fontweight="bold")
    fig.text(-0.04, 0.55, "Extreme Weather Event Combos", 
             va="center", ha="center", rotation="vertical", fontsize=18, fontweight="bold")

    if save_path is not None:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        
    plt.show()

In [ ]:
plot_grid_disturbance_probability_per_percentile_NY_times(
    dfs=[
        all_outage_all_seasons_filtered_90_combined,
         all_undervolt_all_seasons_filtered_90_combined
        ],
    titles=["", ""],
    plot_title="",
    alpha = 0.9, 
    weather_event_order = ['T', 'P', 'W', 'L', 'TP', 'TW', 'TL', 'PW', 'PL', 'PWL'][::-1],
    figsize=(16,6),
    xlim=(0,50), 
    x_label='Probability of Occurrence', 
    show_sig=True,
    show_sig_legend=True,
    duration_legend_pos='lower right',
    duration_legend_anchor=(1.02, 0),
    sig_legend_pos='center right',
    sig_legend_anchor=(1.12, 0.5), 
    # hline_pos=[0], 
    save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/grid_disturbance_prob_6km_90th.png'
)

In [ ]:
plot_grid_disturbance_probability_per_percentile_NY_times(
    dfs=[
        all_outage_all_seasons_filtered_95_combined,
         all_undervolt_all_seasons_filtered_95_combined
        ],
    titles=["", ""],
    plot_title="",
    alpha = 0.9, 
    weather_event_order=['T', 'P', 'W', 'L', 'TW', 'TL', 'PW', 'PL', 'WL', 'PWL'][::-1],
    figsize=(16,6),
    xlim=(0,50), 
    x_label='Probability of Occurrence', 
    show_sig=True,
    show_sig_legend=True,
    duration_legend_pos='lower right',
    duration_legend_anchor=(1.02, 0),
    sig_legend_pos='center right',
    sig_legend_anchor=(1.12, 0.5), 
    # hline_pos=[0], 
    save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/grid_disturbance_prob_6km_95th.png'
)

In [ ]:
plot_grid_disturbance_probability_per_percentile_NY_times(
    dfs=[
        all_outage_all_seasons_filtered_99_combined,
         all_undervolt_all_seasons_filtered_99_combined
        ],
    titles=["", ""],
    plot_title="",
    alpha = 0.9, 
    weather_event_order=['T', 'P', 'W', 'L', 'TW', 'TL', 'PW', 'PL', 'WL', 'PWL'][::-1],
    figsize=(16,6),
    xlim=(0,105), 
    vline_pos=(20, 40, 60, 80),
    x_label='Probability of Occurrence', 
    show_sig=True,
    show_sig_legend=True,
    duration_legend_pos='lower right',
    duration_legend_anchor=(1.02, 0.2),
    sig_legend_pos='center right',
    sig_legend_anchor=(1.12, 0.5), 
    # hline_pos=[0], 
    save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/grid_disturbance_prob_6km_99th.png'
)